In [36]:
import tensorflow as tf
from tensorflow import keras
import os
import pandas as pd
from tensorflow.keras.applications import ResNet50V2

In [11]:
PATH = 'datasets/ISIC_2020_corrected'
os.listdir(PATH)

['ISIC_2020_Training_GroundTruth_v2.csv',
 'subset.csv',
 'train',
 'train_resized',
 'train_split.csv',
 'val_split.csv']

In [64]:
train = pd.read_csv(f'{PATH}/subset.csv')
val = pd.read_csv(f'{PATH}/val_split.csv')

train.shape, val.shape

((2220, 9), (10932, 9))

In [65]:
train.head()

,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_0533349,IP_5208504,IL_9850276,female,45.0,lower extremity,unknown,benign,0
1,ISIC_8814612,IP_0414408,IL_6316968,male,50.0,torso,unknown,benign,0
2,ISIC_6515241,IP_6245507,IL_7464258,male,45.0,lower extremity,unknown,benign,0
3,ISIC_5075261,IP_2117218,IL_8357539,male,40.0,upper extremity,unknown,benign,0
4,ISIC_2624460,IP_1969685,IL_7993831,male,50.0,torso,unknown,benign,0


In [66]:
BATCH_SIZE = 64

def decode(name, label):
    img = tf.io.read_file(name)
    img = tf.image.decode_jpeg(img, channels = 3)
    img = tf.cast(img, tf.float32)
    return img, label
    
def load_ds(df):
    imgs, labels = df["image_name"].values, df["target"].values
    imgs = [f'{PATH}/train_resized/{name}.jpg' for name in imgs]
    ds = tf.data.Dataset.from_tensor_slices((imgs, labels))
    ds = ds.map(decode)
    ds = ds.shuffle(2048)
    ds = ds.batch(BATCH_SIZE)
    return ds

In [67]:
train_ds = load_ds(train)
val_ds = load_ds(val)

In [68]:
IMAGE_SIZE = (256, 256, 3)

encoder = ResNet50V2(
    include_top=False,
    input_shape=IMAGE_SIZE,
    weights='imagenet'
)
encoder.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE)
x = keras.layers.experimental.preprocessing.Rescaling(1./255)(inputs)
x = encoder(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "model_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_20 (InputLayer)       [(None, 256, 256, 3)]     0         
                                                                 
 rescaling_7 (Rescaling)     (None, 256, 256, 3)       0         
                                                                 
 resnet50v2 (Functional)     (None, 8, 8, 2048)        23564800  
                                                                 
 global_average_pooling2d_4   (None, 2048)             0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dense_8 (Dense)             (None, 1)                 2049      
                                                                 
Total params: 23,566,849
Trainable params: 2,049
Non-trainable params: 23,564,800
___________________________________________

In [69]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.AUC(name="auc")]
)

In [70]:
model.fit(train_ds, epochs=10, validation_data=val_ds, validation_steps=10)

Epoch 1/10
35/35 [==============================] - 10s 223ms/step - loss: 0.2158 - auc: 0.4426 - val_loss: 0.0790 - val_auc: 0.4992
Epoch 2/10
35/35 [==============================] - 8s 208ms/step - loss: 0.1135 - auc: 0.4594 - val_loss: 0.1412 - val_auc: 0.5078
Epoch 3/10
35/35 [==============================] - 7s 193ms/step - loss: 0.0971 - auc: 0.5815 - val_loss: 0.0786 - val_auc: 0.5655
Epoch 4/10
35/35 [==============================] - 7s 191ms/step - loss: 0.0863 - auc: 0.6750 - val_loss: 0.0961 - val_auc: 0.6594
Epoch 5/10
35/35 [==============================] - 7s 193ms/step - loss: 0.0787 - auc: 0.7696 - val_loss: 0.0759 - val_auc: 0.7221
Epoch 6/10
35/35 [==============================] - 7s 192ms/step - loss: 0.0734 - auc: 0.8214 - val_loss: 0.1074 - val_auc: 0.7444
Epoch 7/10
35/35 [==============================] - 7s 193ms/step - loss: 0.0702 - auc: 0.8491 - val_loss: 0.1082 - val_auc: 0.8169
Epoch 8/10
35/35 [==============================] - 7s 192ms/step - loss: 0

In [61]:
model.save(r'C:\Users\Usuario\Desktop\Pablo\TFG\PythonEnv\modelos\baselineSubset')

INFO:tensorflow:Assets written to: C:\Users\Usuario\Desktop\Pablo\TFG\PythonEnv\modelos\baselineSubset\assets
